# Vachan V2 — Per-Persona Tone Dial (control vectors)

**What this adds on top of the basic spike:** instead of a generic Hinglish / formal-English contrast, we build the tone vector from a *real persona's own anchors* — the Hinglish phrases we already store for that persona vs. their English-translated equivalents.

**Why it matters:** the basic spike proves the mechanism works. This notebook proves it works *for a specific voice* — so persona A's dial is different from persona B's. That's the Vachan V2 differentiator.

**What you need to run this:**
- A Kaggle account (free) with GPU T4 enabled
- The persona's anchors — paste them into `PERSONA_HINGLISH_ANCHORS` and `PERSONA_ENGLISH_ANCHORS` below. We ship example anchors so it runs out of the box.

> Runtime: ~6–12 min on Kaggle T4. Same model as the basic spike (Llama-3.1-8B-Instruct, 4-bit).

## 0. Setup

1. **Kaggle**: top-right ⋮ → **Accelerator → GPU T4**. Also **Internet: On** (Settings panel).
2. No HuggingFace token needed — we use the same non-gated mirror as the basic spike.
3. Paste your persona's real anchors in Cell 2, or leave the examples to verify the mechanism first.
4. **Run All**.

In [ ]:
!pip install -q repeng transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 1. Persona anchors — paste your real ones here

Each `PERSONA_HINGLISH_ANCHORS[i]` and `PERSONA_ENGLISH_ANCHORS[i]` should be *the same semantic content*, just in different tones. We ship Priya-style example anchors so this runs out of the box — replace with the real persona's stored anchors before the production run.

Rule of thumb: 10–20 pairs → a clean vector. Fewer is fine for a first check.

In [ ]:
# ── PERSONA CONFIG ──────────────────────────────────────────────────────────
PERSONA_NAME = "Priya"   # change to match your persona

# Paste the persona's own Hinglish anchor phrases here.
# These are the POSITIVE end of the dial (casual, warm, code-mixed).
PERSONA_HINGLISH_ANCHORS = [
    "arre yaar deployment ho gaya, tension mat lo",
    "bhai sab theek hai, bas thoda time lagega",
    "haan, main check kar leti hoon abhi",
    "kal tak ready ho jayega, pakka",
    "testing chal rahi hai, aur kuch chahiye?",
    "no worries, fix kar denge",
    "okay so basically issue ye tha...",
    "chill, sab handle ho gaya",
    "thodi der mein bata deti hoon",
    "ekdum sahi track pe hai",
]

# English translations of the SAME content — NEGATIVE end of the dial (formal).
PERSONA_ENGLISH_ANCHORS = [
    "The deployment has been completed successfully. There is no need for concern.",
    "Everything is in order. We will require a bit more time.",
    "Yes, I will check on that immediately.",
    "It will be ready by tomorrow without fail.",
    "Testing is currently in progress. Do you require anything else?",
    "No concerns — this will be resolved.",
    "To summarize, the root cause of the issue was as follows.",
    "The situation is under control and has been handled.",
    "I will update you shortly.",
    "The project is fully on track.",
]

assert len(PERSONA_HINGLISH_ANCHORS) == len(PERSONA_ENGLISH_ANCHORS), "Lists must be same length"
print(f"Persona: {PERSONA_NAME} | {len(PERSONA_HINGLISH_ANCHORS)} anchor pairs loaded")

## 2. Build contrastive dataset from anchors

For each anchor pair we generate multiple chat-template prefixes (different user messages) so repeng sees the vector across many token positions — not just one sentence. This makes the extracted direction more robust.

In [ ]:
# These user messages give the model a reason to use the anchor tone.
# Diverse prompts → the vector generalises across topic, not just phrasing.
USER_PROMPTS = [
    "Can you give me an update on the deployment?",
    "How is the project going?",
    "Any blockers right now?",
    "What's the status?",
    "Is everything on track?",
    "Let me know when it's ready.",
    "What happened with the issue?",
    "Do we need to do anything?",
]

def make_entry(hinglish_anchor: str, english_anchor: str, user_msg: str) -> DatasetEntry:
    """
    positive = system prompt that places the persona's Hinglish voice, then user msg,
               then the anchor as the assistant's start (partial assistant turn).
    negative = same structure but English anchor.
    repeng reads hidden states at each token of the partial assistant turn.
    """
    def build(anchor: str) -> str:
        msgs = [
            {
                "role": "system",
                "content": (
                    f"You are {PERSONA_NAME}, a warm and helpful assistant."
                ),
            },
            {"role": "user", "content": user_msg},
        ]
        base = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return base + anchor  # anchor is the partial assistant turn

    return DatasetEntry(positive=build(hinglish_anchor), negative=build(english_anchor))

dataset = [
    make_entry(h, e, u)
    for h, e in zip(PERSONA_HINGLISH_ANCHORS, PERSONA_ENGLISH_ANCHORS)
    for u in USER_PROMPTS
]

print(f"{len(dataset)} contrastive entries ({len(PERSONA_HINGLISH_ANCHORS)} anchors × {len(USER_PROMPTS)} user msgs)")
print("--- sample positive (tail 200 chars) ---")
print(dataset[0].positive[-200:])

## 3. Extract the per-persona control vector

Same as the basic spike — one forward pass, no gradients. The result is a per-layer direction that encodes *this persona's* Hinglish vs English axis.

In [ ]:
model.reset()
persona_vector = ControlVector.train(model, tokenizer, dataset)

_layers = list(persona_vector.directions.keys())
print(f"{PERSONA_NAME} control vector: {len(_layers)} layers, shape {persona_vector.directions[_layers[0]].shape}")

## 4. Turn the dial — same prompt, three tones

In [ ]:
def generate(prompt: str, coeff: float) -> str:
    model.reset()
    if coeff != 0:
        model.set_control(persona_vector, coeff)
    msgs = [
        {"role": "system", "content": f"You are {PERSONA_NAME}, a warm and helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    ids = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(
        ids,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()

PROMPT = "Can you give me a status update on the project?"
for c in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    print(f"\n=========== coeff {c:+} ===========")
    print(generate(PROMPT, c))

## 5. Save the vector

Once the dial works, save the vector so it can be reused without rerunning this notebook. The `.pt` file is the artifact — load it at inference time to apply the tone dial.

In [ ]:
import os, torch

OUT_DIR = f"./persona_vectors"
os.makedirs(OUT_DIR, exist_ok=True)

out_path = os.path.join(OUT_DIR, f"{PERSONA_NAME.lower()}_tone_vector.pt")
torch.save(persona_vector.directions, out_path)

print(f"Saved: {out_path}")
print("To reload:")
print(f"  directions = torch.load('{out_path}')")
print(f"  vec = ControlVector(model_type=model.config.model_type, directions=directions)")

## 6. What's next (V2 path)

**If the dial moves with the coeff → this persona's vector is proven.** Now you have a `.pt` file that is that persona's reusable tone dial.

**Tuning if the effect is weak:**
- Add more anchor pairs (aim for 15–20+)
- Widen the layer band: `range(-3, -22, -1)`
- Try `coeff` values up to ±3, but back off if text degenerates

**Path into Vachan production (tracked separately):**
1. Store the `.pt` per persona in your artifact store
2. Spin up a vLLM / transformers inference endpoint that loads the base model + injects the vector at generation time
3. Add a `tone_coeff` parameter to the call — PFS gate passes it based on fidelity score
4. The Fidelity Ring's `av_cosine` becomes the objective you tune `coeff` against (baseline: 0.71 from PR #2)

> This endpoint is only needed for high-value personas that the PFS gate keeps failing — the majority still go through Groq.